# Vizualizime Interaktive - NYC Airbnb 2019

Ky notebook shton pjesën bonus të vizualizimit interaktiv me Plotly dhe Folium. Output-et ruhen në `outputs/interactive/`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import folium
from folium.plugins import MarkerCluster

sys.path.append("../src")
from airbnb_utils import add_analysis_features, create_summary_tables

DATA_PATH = Path("../data/cleaned_airbnb.csv")
OUTPUT_DIR = Path("../outputs/interactive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K = pd.read_csv(DATA_PATH)
K = add_analysis_features(K)
K.head()


## Grafik interaktiv: Çmimi sipas zonës dhe tipit të dhomës

Ky grafik ndihmon krahasimin e çmimeve mes zonave dhe tipave të dhomave. Interaktiviteti lejon filtrimin dhe leximin e vlerave pa e mbingarkuar figurën.


In [ ]:
sample = K.sample(min(8000, len(K)), random_state=42)

fig = px.box(
    sample,
    x="neighbourhood_group",
    y="price",
    color="room_type",
    points="outliers",
    title="Çmimi sipas zonës dhe tipit të dhomës",
    labels={
        "neighbourhood_group": "Zona",
        "price": "Çmimi ($)",
        "room_type": "Tipi i dhomës",
    },
)
fig.write_html(OUTPUT_DIR / "price_by_area_room_type.html")
fig.show()


## Hartë interaktive Folium

Për performancë përdoret një mostër e kontrolluar e listimeve. Ngjyra përfaqëson tipin e dhomës, ndërsa popup-i shfaq zonën, lagjen dhe çmimin.


In [ ]:
map_sample = K.dropna(subset=["latitude", "longitude"]).sample(min(2500, len(K)), random_state=7)

room_colors = {
    "Entire home/apt": "red",
    "Private room": "blue",
    "Shared room": "green",
}

m = folium.Map(
    location=[map_sample["latitude"].mean(), map_sample["longitude"].mean()],
    zoom_start=11,
    tiles="cartodbpositron",
)
cluster = MarkerCluster().add_to(m)

for _, row in map_sample.iterrows():
    popup = (
        f"<b>{row['neighbourhood_group']}</b><br>"
        f"{row['neighbourhood']}<br>"
        f"{row['room_type']}<br>"
        f"Price: ${row['price']}"
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=room_colors.get(row["room_type"], "gray"),
        fill=True,
        fill_opacity=0.65,
        popup=popup,
    ).add_to(cluster)

m.save(OUTPUT_DIR / "airbnb_listing_map.html")
m


## Interpretim

- Manhattan dhe Brooklyn kanë dendësi më të madhe listimesh.
- Tipi `Entire home/apt` zakonisht ka çmim më të lartë se `Private room`.
- Harta interaktive e bën më të lehtë identifikimin e zonave me përqendrim të lartë listimesh dhe çmimesh.
